In [33]:
import pandas as pd
import numpy as np

In [34]:
import os
import pandas as pd

# Check if the code is running inside Google Colab
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    # Point to your Drive folder for the cloud
    file_path = '/content/drive/MyDrive/metadata.csv'
else:
    # Point to the local folder for GitHub and local testing
    file_path = 'metadata.csv'

df = pd.read_csv(file_path)
print("Data loaded successfully!")

Data loaded successfully!


In [35]:
df = pd.read_csv(file_path)
df.head()


,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,TRUE,TRUE,TRUE,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,FALSE,TRUE,TRUE,PAT_684_1302_588.png,True


In [36]:


# Assuming you've already loaded your CSV into a DataFrame called 'df'
# df = pd.read_csv('pad_ufes_20.csv')

# Define the mapping dictionary based on your project rules
label_mapping = {
    'ACK': 0, 'SEK': 0, 'NEV': 0,             # Benign
    'MEL': 1, 'BCC': 1, 'SCC': 1, 'BOW': 1    # Malignant
}

# Apply the mapping directly to the existing column
df['diagnostic'] = df['diagnostic'].map(label_mapping)

# Quick sanity check to ensure the mapping worked correctly
print(df['diagnostic'].value_counts(dropna=False))

diagnostic
0    1209
1    1089
Name: count, dtype: int64


In [37]:
#Set the image filename as the index of the dataframe
# This keeps it attached to the row forever, but hides it from XGBoost
df = df.set_index('img_id')
# 1. Convert 'UNK' strings into actual missing values (NaN)
df = df.replace('UNK', np.nan)

# 2. Apply the boolean mapping and silence the pandas warning
bool_map = {'TRUE': 1, 'FALSE': 0, 'True': 1, 'False': 0, True: 1, False: 0}
df = df.replace(bool_map).infer_objects(copy=False)

# 3. Force columns that contain numbers (and NaNs) to be recognized as numeric floats.
# (pandas requires columns with NaNs to be floats, not integers)
df = df.apply(pd.to_numeric, errors='ignore')

# 4. Re-run the correlation matrix
correlations = df.corr(numeric_only=True)['diagnostic'].sort_values(ascending=False)

print("Updated Correlation with Diagnostic:")
print(correlations)

Updated Correlation with Diagnostic:
diagnostic             1.000000
biopsed                0.801039
bleed                  0.445187
grew                   0.397079
elevation              0.372697
hurt                   0.357653
changed                0.229372
age                    0.210915
itch                   0.200698
smoke                  0.109027
diameter_2             0.101893
diameter_1             0.063029
pesticide              0.060983
skin_cancer_history    0.041143
drink                  0.037600
cancer_history        -0.005502
has_piped_water       -0.021038
has_sewage_system     -0.038473
fitspatrick           -0.133017
lesion_id             -0.438350
Name: diagnostic, dtype: float64


C:\Users\HP\AppData\Local\Temp\ipykernel_17476\888603882.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(bool_map).infer_objects(copy=False)
C:\Users\HP\AppData\Local\Temp\ipykernel_17476\888603882.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')


In [ ]:
# 1. Create a new single column that takes the maximum value of the two
df['diameter'] = df[['diameter_1', 'diameter_2']].max(axis=1)


# 2. Add the original two columns to your drop list
cols_to_drop = [
    'patient_id', 
    'lesion_id', 
    'has_piped_water', 
    'has_sewage_system', 
    'biopsed', 
    'cancer_history', 
    'drink', 
    'pesticide',
    'diameter_1',
    'diameter_2',
    'background_father',
    'background_mother'
    
]

# 3. Drop them all at once
df = df.drop(columns=cols_to_drop, errors='ignore')


In [40]:
# 2. Map gender to 0 and 1
df['gender'] = df['gender'].map({'FEMALE': 0, 'MALE': 1})

# 3. One-hot encode the region column
# dtype=int forces the output to be 1 and 0 instead of True and False
df = pd.get_dummies(df, columns=['region'], dtype=int)
df.head()

,smoke,age,gender,skin_cancer_history,fitspatrick,diagnostic,itch,grew,hurt,changed,...,region_EAR,region_FACE,region_FOOT,region_FOREARM,region_HAND,region_LIP,region_NECK,region_NOSE,region_SCALP,region_THIGH
img_id,,,,,,,,,,,,,,,,,,,,,
PAT_1516_1765_530.png,NaN,8,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
PAT_46_881_939.png,0.0,55,0.0,1.0,3.0,1,1.0,1.0,0.0,1.0,...,0,0,0,0,0,0,1,0,0,0
PAT_1545_1867_547.png,NaN,77,NaN,NaN,NaN,0,1.0,0.0,0.0,0.0,...,0,1,0,0,0,0,0,0,0,0
PAT_1989_4061_934.png,NaN,75,NaN,NaN,NaN,0,1.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
PAT_684_1302_588.png,0.0,79,1.0,1.0,1.0,1,1.0,1.0,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0


,smoke,age,gender,skin_cancer_history,fitspatrick,diagnostic,itch,grew,hurt,changed,bleed,elevation,diameter,region_ABDOMEN
img_id,,,,,,,,,,,,,,
PAT_1516_1765_530.png,NaN,8,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
PAT_46_881_939.png,0.0,55,0.0,1.0,3.0,1,1.0,1.0,0.0,1.0,1.0,1.0,6.0,0
PAT_1545_1867_547.png,NaN,77,NaN,NaN,NaN,0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,0
PAT_1989_4061_934.png,NaN,75,NaN,NaN,NaN,0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,0
PAT_684_1302_588.png,0.0,79,1.0,1.0,1.0,1,1.0,1.0,0.0,0.0,1.0,1.0,5.0,0
